In [ ]:
import numpy as np
import math
from scipy.constants import speed_of_light  
#近场阵列流形计算

#LOS路径建模

#多簇多径NLOS环境

#适当的路径损耗和阴影效应

#加性高斯白噪声


def nearfield_array_manifold(theta, r, M, d, wavelength):                                                      
    #计算近场阵列流形向量
    
    a = np.zeros((M, 1), dtype=complex)
    for m in range(M):
        # 论文中的公式(3)
        phase_term1 = -2 * np.pi * d * m * np.sin(theta) / wavelength
        phase_term2 = np.pi * (d**2) * (m**2) * (np.cos(theta)**2) / (wavelength * r)
        phase = phase_term1 + phase_term2
        a[m] = np.exp(1j * phase)
    return a

def calculate_aoa(true_pos, ap_pos):
    
    #计算到达角 (AoA) - 遵循论文公式(4)和(7)
    
    delta_x = true_pos[0] - ap_pos[0]
    delta_y = true_pos[1] - ap_pos[1]
    
    # 计算基本角度
    base_angle = np.arctan2(delta_y, delta_x)
    
    # 调整角度到正确的象限
    if delta_x < 0:
        adjusted_angle = base_angle + np.pi
    else:
        adjusted_angle = base_angle
    
    return adjusted_angle

def calculate_distance(true_pos, ap_pos):
    
    #计算距离 - 遵循论文公式(5)和(8)
    
    return np.sqrt((true_pos[0] - ap_pos[0])**2 + (true_pos[1] - ap_pos[1])**2)

def path_loss(distance, fc, scenario='indoor'):
    """
    路径损耗模型 (基于WINNER II室内模型)
    """
    d0 = 1.0  # 参考距离 1m
    PL0 = 20 * np.log10(4 * np.pi * d0 * fc / speed_of_light)  
    n = 3.0  # 室内路径损耗指数
    return PL0 + 10 * n * np.log10(max(distance, d0)/d0)

def generate_ap_signal(true_pos, ap_pos, M, d, fc, num_clusters=3, paths_per_cluster=3, SNR_dB=20):
    """
    为单个AP生成接收信号
    """
    wavelength = speed_of_light / fc  
    
    # 计算LOS路径参数
    r_los = calculate_distance(true_pos, ap_pos)
    theta_los = calculate_aoa(true_pos, ap_pos)
    
    # 计算LOS路径的阵列流形向量
    a_los = nearfield_array_manifold(theta_los, r_los, M, d, wavelength)
    
    # 计算LOS路径损耗
    PL_los = path_loss(r_los, fc)
    
    # 初始化接收信号
    y_p = np.zeros((M, 1), dtype=complex)
    
    # LOS分量
    los_gain = np.sqrt(10**(-PL_los/10)) * np.exp(1j * 2 * np.pi * np.random.rand())
    y_p += a_los * los_gain
    
    # NLOS分量 (簇)
    for c in range(num_clusters):
        # 随机生成簇的中心角度 (在LOS角度附近±60度范围内)
        cluster_angle = theta_los + np.random.uniform(-np.pi/3, np.pi/3)
        
        # 随机生成额外的路径长度 (5-20米)
        extra_distance = np.random.uniform(5, 20)
        cluster_distance = r_los + extra_distance
        
        # 簇内路径，
        for p in range(paths_per_cluster):
            # 路径角度稍微偏离簇中心 (±15度)
            path_angle = cluster_angle + np.random.uniform(-np.pi/12, np.pi/12)
            
            # 路径距离稍微偏离簇距离
            path_distance = cluster_distance + np.random.uniform(-2, 2)
            
            # 计算路径损耗
            PL_path = path_loss(path_distance, fc)
            
            # 计算阵列流形向量
            a_path = nearfield_array_manifold(path_angle, path_distance, M, d, wavelength)
            
            # 路径增益 (复随机变量)
            path_gain = np.sqrt(10**(-PL_path/10)) * (np.random.randn() + 1j * np.random.randn()) / np.sqrt(2)
            
            # 添加到接收信号
            y_p += a_path * path_gain
    
    # 添加噪声
    signal_power = np.mean(np.abs(y_p)**2)
    noise_power = signal_power / (10**(SNR_dB/10))
    noise = np.sqrt(noise_power/2) * (np.random.randn(M, 1) + 1j * np.random.randn(M, 1))
    y_p += noise
    
    return y_p

def generate_all_ap_signals(true_pos, ap_positions, params):

    all_signals = []
    
    for i, ap_pos in enumerate(ap_positions):
        print(f"Generating signal for AP {i+1} at position {ap_pos}...")
        y_p = generate_ap_signal(
            true_pos=true_pos,
            ap_pos=ap_pos,
            M=params['M'],
            d=params['d'],
            fc=params['fc'],
            num_clusters=params['num_clusters'],
            paths_per_cluster=params['paths_per_cluster'],
            SNR_dB=params['SNR_DB']
        )
        all_signals.append(y_p)
    
    return all_signals

# 文献参数设置 

# 文献中的AP位置 (单位: 米)
ap_positions = [
    [0.0, 20.0],   # AP1
    [20.0, 15.0],  # AP2  
    [12.0, 0.0]    # AP3
]

# 文献用户位置
true_position = [7.0, 8.0]

# 系统参数
params = {
    'M': 10,                    # 每个AP的天线数量
    'd': 0.087,                 # 天线间距 
    'fc': 2.4e9,                # 载波频率
    'bandwidth': 20e6,          # 带宽
    'num_clusters': 3,          # NLOS簇数量  有时候我会加到5
    'paths_per_cluster': 3,     # 每个簇的路径数
    'SNR_DB': 20                # 信噪比
}

# 计算波长
params['wavelength'] = speed_of_light / params['fc']

print("WINNER II 信道模型仿真 ")
print(f"用户位置: {true_position}")
print(f"AP位置: {ap_positions}")
print(f"天线数: {params['M']}")
print(f"天线间距: {params['d']} m")
print(f"载波频率: {params['fc']/1e9} GHz")
print(f"波长: {params['wavelength']:.4f} m")
print(f"SNR: {params['SNR_DB']} dB")

# 为所有AP生成接收信号
all_signals = generate_all_ap_signals(true_position, ap_positions, params)

# 输出结果

for i, signal in enumerate(all_signals):
    print(f"AP{i+1} 接收信号形状: {signal.shape}")
    print(f"AP{i+1} 接收信号功率: {10*np.log10(np.mean(np.abs(signal)**2)):.2f} dBm")
    
    # 计算该AP的LOS参数（用于验证）
    r_los = calculate_distance(true_position, ap_positions[i])
    theta_los = calculate_aoa(true_position, ap_positions[i])
    print(f"AP{i+1} LOS距离: {r_los:.2f} m, LOS角度: {np.degrees(theta_los):.2f}°")
    print("-" * 50)

# 信号可以用于后续的定位算法
y_p1, y_p2, y_p3 = all_signals
print(all_signals)

WINNER II 信道模型仿真 
用户位置: [7.0, 8.0]
AP位置: [[0.0, 20.0], [20.0, 15.0], [12.0, 0.0]]
天线数: 10
天线间距: 0.087 m
载波频率: 2.4 GHz
波长: 0.1249 m
SNR: 20 dB
Generating signal for AP 1 at position [0.0, 20.0]...
Generating signal for AP 2 at position [20.0, 15.0]...
Generating signal for AP 3 at position [12.0, 0.0]...
AP1 接收信号形状: (10, 1)
AP1 接收信号功率: -71.45 dBm
AP1 LOS距离: 13.89 m, LOS角度: -59.74°
--------------------------------------------------
AP2 接收信号形状: (10, 1)
AP2 接收信号功率: -70.42 dBm
AP2 LOS距离: 14.76 m, LOS角度: 28.30°
--------------------------------------------------
AP3 接收信号形状: (10, 1)
AP3 接收信号功率: -67.48 dBm
AP3 LOS距离: 9.43 m, LOS角度: 302.01°
--------------------------------------------------
[array([[ 4.54181389e-06+8.20659746e-05j],
       [ 4.38084643e-05-3.25201332e-04j],
       [ 4.90912566e-05-7.34477325e-05j],
       [ 1.60863576e-04-1.76817579e-04j],
       [-4.01322382e-04-8.62682954e-05j],
       [-1.59883045e-04+1.03979527e-05j],
       [-1.65042214e-04-3.33823484e-04j],
       [-1.2492